# 02 · The bus in your own model (plain Hugging Face + forward hooks)

AI4Read post 03. Notebook 01 used TransformerLens on GPT-2. This one uses only `transformers` and PyTorch forward hooks,
so it works for the models you load yourself. The question it answers: **is the vector `output_hidden_states` returns at
index L the residual stream after L blocks, i.e. the embedding plus every attention and MLP write before it?**

Steps (card §10): 1 module tree, config and decoder-layer source · 2 `hidden_states[0]` versus the embedding lookup ·
3 forward hooks on the attention and MLP modules *found by inspection* · 4 per-block additivity against
`hidden_states`, with a report of where it fails · 5 the last entry versus the final norm · 6 one explicit position and
layer, rebuilt from its writes · 7 arithmetic on a real stream (a random unit direction) · 8 the same checks on tiny
randomly initialised models built from a config only · 9 blank exercises.

Float32 on the CPU for every equality check (card T7). Library behaviour is never assumed: it is printed from the
installed source with file and line (`cite`, `show_source`) or shown numerically.

## 0 · Parameters

* `MODEL_PATH`: a Hugging Face id or a local folder. Default `Qwen/Qwen3-1.7B-Base`; if it cannot be loaded the
  notebook falls back to `FALLBACK_MODEL` and says so. (Environment variable `AI4READ_MODEL_PATH` overrides it.)
* `LOCAL_ONLY = True`: never touch the network (needs the model in the Hugging Face cache or a local folder).
  Environment variable `AI4READ_LOCAL_ONLY=1` overrides it.
* `PROMPT`, `POS` (token position, negative counts from the end), `LAYER` (number of blocks applied; `None` = half way).

In [1]:
import os
MODEL_PATH = "Qwen/Qwen3-1.7B-Base"
FALLBACK_MODEL = "openai-community/gpt2"
LOCAL_ONLY = False
PROMPT = "The cat sat on the mat because"
POS = -1          # explicit token position (-1 = last token)
LAYER = None      # explicit layer = number of blocks applied (None = n_layers // 2)
TINY_CONFIG_FROM = "Qwen/Qwen3-1.7B-Base"   # section 8: only its config.json is used
SEED = 0

MODEL_PATH = os.environ.get("AI4READ_MODEL_PATH", MODEL_PATH)
LOCAL_ONLY = os.environ.get("AI4READ_LOCAL_ONLY", str(LOCAL_ONLY)).lower() in ("1", "true", "yes")
if LOCAL_ONLY:                       # must be set before transformers / huggingface_hub are imported
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("MODEL_PATH =", MODEL_PATH, "| LOCAL_ONLY =", LOCAL_ONLY)

MODEL_PATH = openai-community/gpt2 | LOCAL_ONLY = True


In [2]:
import os, re, math, pathlib, platform, importlib, importlib.metadata as md

def pkg_version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return "not installed"

def cite(module, pattern, context=0, max_hits=1, start_after=None):
    """Print file:line for lines matching `pattern` in the INSTALLED source of `module`.
    Line numbers are those of the version installed on this machine."""
    if isinstance(module, str):
        try:
            module = importlib.import_module(module)
        except Exception as e:  # a different library version may have moved the file
            print(f"[module {module} not importable here ({type(e).__name__}); the installed version may differ]")
            return
    path = pathlib.Path(module.__file__)
    parts = path.parts
    cut = max([i for i, p in enumerate(parts) if p in ("site-packages", "dist-packages")] or [0])
    rel = "/".join(parts[cut + 1:])
    lines = path.read_text().splitlines()
    begin = 0
    if start_after is not None:
        begin = next((i for i, l in enumerate(lines) if re.search(start_after, l)), 0)
    hits = [i for i in range(begin, len(lines)) if re.search(pattern, lines[i])][:max_hits]
    if not hits:
        print(f"[pattern not found in {rel}: {pattern!r}; the installed version may differ]")
    for i in hits:
        for j in range(max(0, i - context), min(len(lines), i + context + 1)):
            print(f"{rel}:{j + 1}: {lines[j].rstrip()}")

def err(a, b):
    """(max absolute error, max relative error). Relative = max over positions of
    ||a - b||_2 / ||b||_2, i.e. relative to the norm of the second argument b (the reference) at that position."""
    d = (a - b).double()
    max_abs = d.abs().max().item()
    rel = (d.norm(dim=-1) / b.double().norm(dim=-1).clamp_min(1e-30)).max().item()
    return max_abs, rel

import inspect, itertools, numpy as np, pandas as pd, torch, transformers
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
torch.set_grad_enabled(False)
transformers.utils.logging.disable_progress_bar()   # keep saved outputs small
DEVICE, DTYPE = "cpu", torch.float32
pd.set_option("display.width", 160)
print("python      ", platform.python_version(), "|", platform.platform())
for p in ["torch", "transformers", "numpy", "pandas", "huggingface_hub", "safetensors", "tokenizers"]:
    print(f"{p:12s}", pkg_version(p))
print("device", DEVICE, "| dtype", DTYPE)

def show_source(obj, pattern=None):
    """Print the installed source of a function/class with file:line numbers (optionally only matching lines)."""
    obj = inspect.unwrap(obj)               # skip decorators, so file and lines belong to the same function
    lines, start = inspect.getsourcelines(obj)
    path = pathlib.Path(inspect.getsourcefile(obj)).parts
    cut = max([i for i, p in enumerate(path) if p in ("site-packages", "dist-packages")] or [0])
    rel = "/".join(path[cut + 1:])
    for i, l in enumerate(lines):
        if pattern is None or re.search(pattern, l):
            print(f"{rel}:{start + i}: {l.rstrip()}")

python       3.13.16 | Linux-6.18.44-fc-v153-x86_64-with-glibc2.39
torch        2.14.1
transformers 5.19.0
numpy        2.5.3
pandas       3.0.5
huggingface_hub 2.2.0
safetensors  0.8.0
tokenizers   0.23.3
device cpu | dtype torch.float32


## 1 · Load (with fallback), then inspect

The model is loaded in float32 with `attn_implementation="eager"`. If `MODEL_PATH` fails (not downloaded, no network
with `LOCAL_ONLY`, not enough memory), the fallback is loaded and the reason is printed.

In [3]:
def load(name):
    tok = AutoTokenizer.from_pretrained(name, local_files_only=LOCAL_ONLY)
    m = AutoModelForCausalLM.from_pretrained(name, dtype=DTYPE, attn_implementation="eager",
                                             local_files_only=LOCAL_ONLY).to(DEVICE).eval()
    return m, tok

try:
    model, tok = load(MODEL_PATH)
    LOADED = MODEL_PATH
except Exception as e:
    print(f"Could not load {MODEL_PATH!r}: {type(e).__name__}: {str(e)[:300]}")
    print(f"Falling back to {FALLBACK_MODEL!r}.")
    model, tok = load(FALLBACK_MODEL)
    LOADED = FALLBACK_MODEL
print("loaded:", LOADED, "|", type(model).__name__, "| parameters:", f"{sum(p.numel() for p in model.parameters()):,}",
      "| dtype:", next(model.parameters()).dtype)

loaded: openai-community/gpt2 | GPT2LMHeadModel | parameters: 124,439,808 | dtype: torch.float32


### 1a · Module tree and the pieces found by inspection

No module name is assumed. The block list is the `ModuleList` whose length equals `num_hidden_layers`; the final norm is
a direct child of the base model whose class name contains "Norm"; inside one block, the attention and MLP modules are
guessed from the children's names and classes (norm modules excluded) and then **checked numerically** in section 4.

In [4]:
def find_parts(model):
    cfg = model.config
    n = cfg.num_hidden_layers
    lists = [(name, m) for name, m in model.named_modules() if isinstance(m, torch.nn.ModuleList) and len(m) == n]
    assert lists, "no ModuleList with num_hidden_layers entries"
    layers_name, layers = lists[0]
    base = model.base_model
    norms = [(name, m) for name, m in base.named_children() if "norm" in type(m).__name__.lower()]
    kids = list(layers[0].named_children())
    is_norm = lambda m: "norm" in type(m).__name__.lower()
    attn = [nm for nm, m in kids if not is_norm(m) and re.search(r"attn|attention", nm + type(m).__name__, re.I)]
    mlp = [nm for nm, m in kids if not is_norm(m) and re.search(r"mlp|feed_?forward|ffn", nm + type(m).__name__, re.I)]
    return dict(n_layers=n, layers_name=layers_name, layers=layers, base=base,
                final_norm_name=norms[0][0] if norms else None, final_norm=norms[0][1] if norms else None,
                children=[(nm, type(m).__name__) for nm, m in kids],
                attn_name=attn[0] if len(attn) == 1 else None, mlp_name=mlp[0] if len(mlp) == 1 else None,
                all_block_lists=[nm for nm, _ in lists])

def describe(model, parts):
    cfg = model.config
    base = parts["base"]
    pos_emb = [(nm, type(m).__name__) for nm, m in base.named_children()
               if isinstance(m, torch.nn.Embedding) and m is not model.get_input_embeddings()]
    rotary = [(nm, type(m).__name__) for nm, m in base.named_modules() if "rotary" in type(m).__name__.lower()]
    norm_classes = sorted({type(m).__name__ for _, m in parts["layers"][0].named_children() if "norm" in type(m).__name__.lower()})
    tied = model.get_output_embeddings().weight.data_ptr() == model.get_input_embeddings().weight.data_ptr()
    info = {
        "model_type": cfg.model_type,
        "hidden_size (d)": cfg.hidden_size,
        "num_hidden_layers": cfg.num_hidden_layers,
        "vocab_size": cfg.vocab_size,
        "block list": parts["layers_name"],
        "children of one block": parts["children"],
        "attention module (guess)": parts["attn_name"],
        "MLP module (guess)": parts["mlp_name"],
        "norm classes in a block": norm_classes,
        "final norm": (parts["final_norm_name"], type(parts["final_norm"]).__name__),
        "norm eps (config)": getattr(cfg, "rms_norm_eps", getattr(cfg, "layer_norm_epsilon", getattr(cfg, "layer_norm_eps", None))),
        "position embedding module added to the stream": pos_emb or "none",
        "rotary module": rotary[:1] or "none",
        "config.tie_word_embeddings": getattr(cfg, "tie_word_embeddings", None),
        "lm_head shares storage with the embedding": tied,
        "config.tie_last_hidden_states": getattr(cfg, "tie_last_hidden_states", None),
    }
    for k, v in info.items():
        print(f"{k:48s} {v}")

parts = find_parts(model)
print(model if sum(p.numel() for p in model.parameters()) < 5e9 else type(model))
print()
describe(model, parts)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

model_type         

### 1b · The decoder-layer source and the base model's forward (installed version)

Read these lines before trusting any hook: they show what is added to the stream (`residual + ...`), whether a norm
sits between a sub-layer and the add (card T2), whether the embedding is scaled (T3), and where the final norm is applied.

In [5]:
show_source(type(parts["layers"][0]).forward)
print()
show_source(type(parts["base"]).forward,
            pattern=r"embed|position|rotary|for .* in .*(layer|self\.h)|layer\(|block\(|norm|ln_f|drop|hidden_states = |return")
print()
mod = inspect.getmodule(type(parts["base"]))
cite(mod, r'"hidden_states":', context=0)
cite(mod, r"_tied_weights_keys")

transformers/models/gpt2/modeling_gpt2.py:262:     def forward(
transformers/models/gpt2/modeling_gpt2.py:263:         self,
transformers/models/gpt2/modeling_gpt2.py:264:         hidden_states: tuple[torch.FloatTensor] | None,
transformers/models/gpt2/modeling_gpt2.py:265:         past_key_values: Cache | None = None,
transformers/models/gpt2/modeling_gpt2.py:266:         attention_mask: torch.FloatTensor | None = None,
transformers/models/gpt2/modeling_gpt2.py:267:         encoder_hidden_states: torch.Tensor | None = None,
transformers/models/gpt2/modeling_gpt2.py:268:         encoder_attention_mask: torch.FloatTensor | None = None,
transformers/models/gpt2/modeling_gpt2.py:269:         use_cache: bool | None = False,
transformers/models/gpt2/modeling_gpt2.py:270:         **kwargs,
transformers/models/gpt2/modeling_gpt2.py:271:     ) -> torch.Tensor:
transformers/models/gpt2/modeling_gpt2.py:272:         residual = hidden_states
transformers/models/gpt2/modeling_gpt2.py:273:         

### 1c · How `output_hidden_states` is collected (installed `transformers`)

In this version the tuple is filled by forward hooks on the block class named in `_can_record_outputs` (printed above).
The lines below say what goes in at index 0 and what replaces the last entry.

In [6]:
cite("transformers.utils.output_capturing", r"Historically, the first hidden_states output is actually the input of layer 0", context=2)
cite("transformers.utils.output_capturing", r"collected_outputs\[key\].append\(args\[0\]\)")
cite("transformers.utils.output_capturing", r"if not isinstance\(output, tuple\):", context=3)
cite("transformers.utils.output_capturing", r"Whether to overwrite `out.hidden_states\[-1\]` with the `out.last_hidden_state`", context=3)
cite("transformers.utils.output_capturing", r"collected_outputs\[key\].append\(outputs.last_hidden_state\)", context=1)

transformers/utils/output_capturing.py:113:         hidden_states_layers = collected_outputs.get("_hidden_states_layers")
transformers/utils/output_capturing.py:114: 
transformers/utils/output_capturing.py:115:         # Historically, the first hidden_states output is actually the input of layer 0 (or inputs_embeds), so the hidden_states outputs
transformers/utils/output_capturing.py:116:         # have length `num_hidden_layers + 1`. When we request a list of explicit layers to capture, do not add this into the result, so
transformers/utils/output_capturing.py:117:         # that the indexing of the output is the same as the actual requested layers
transformers/utils/output_capturing.py:124:             collected_outputs[key].append(args[0])
transformers/utils/output_capturing.py:130:                 collected_outputs[key].append(None)
transformers/utils/output_capturing.py:131:                 return
transformers/utils/output_capturing.py:132: 
transformers/utils/output_capturing.py:

## 2 · Tokens, `hidden_states`, and the embedding lookup

Card T5: one stream per position. The cell prints the tokens, whether a BOS token was added, the padding side (a single
unpadded prompt here), and the explicit position used later. Card T3/T4: `hidden_states[0]` is compared with the
embedding-table lookup, and, if that fails, with lookup + a position embedding.

In [7]:
from collections import defaultdict

def first_tensor(x):
    if torch.is_tensor(x):
        return x
    if isinstance(x, (tuple, list)):
        for y in x:
            if torch.is_tensor(y):
                return y
    return None

def collect(model, parts, ids):
    """One forward pass with output_hidden_states=True and forward hooks on every block and every direct child of
    every block. Returns the HF output and a record of inputs/outputs."""
    rec = dict(block_in=[], block_out=[], child_in=defaultdict(dict), child_out=defaultdict(dict))
    handles = []
    for L, layer in enumerate(parts["layers"]):
        def blk_hook(mod, args, kwargs, out, L=L):
            x = args[0] if args else kwargs.get("hidden_states")
            rec["block_in"].append(x.detach().clone()); rec["block_out"].append(first_tensor(out).detach().clone())
        handles.append(layer.register_forward_hook(blk_hook, with_kwargs=True))
        for nm, child in layer.named_children():
            def kid_hook(mod, args, kwargs, out, L=L, nm=nm):
                x = args[0] if args and torch.is_tensor(args[0]) else kwargs.get("hidden_states")
                if torch.is_tensor(x):
                    rec["child_in"][L][nm] = x.detach().clone()
                y = first_tensor(out)
                if y is not None:
                    rec["child_out"][L][nm] = y.detach().clone()
            handles.append(child.register_forward_hook(kid_hook, with_kwargs=True))
    try:
        out = model(ids, output_hidden_states=True)
    finally:
        for h in handles:
            h.remove()
    return out, rec

enc = tok(PROMPT, return_tensors="pt")
ids = enc.input_ids.to(DEVICE)
str_toks = tok.convert_ids_to_tokens(ids[0])
pos = POS % ids.shape[1]
print("token ids  :", ids[0].tolist())
print("tokens     :", str_toks)
print("BOS token  :", tok.bos_token, "| first id is BOS:", tok.bos_token_id is not None and ids[0, 0].item() == tok.bos_token_id)
print("padding    :", tok.padding_side, "(one prompt, no padding)")
print("explicit position POS =", pos, repr(str_toks[pos]))

out, rec = collect(model, parts, ids)
hs = out.hidden_states
n = parts["n_layers"]
print(f"\nlen(hidden_states) = {len(hs)} for {n} blocks; each entry has shape {tuple(hs[0].shape)}")

def embedding_check(model, parts, ids, hs0):
    E = model.get_input_embeddings()
    rows = [("raw table rows E.weight[ids]", *err(hs0, E.weight[ids])),
            ("embedding module E(ids)", *err(hs0, E(ids)))]
    for nm, m in parts["base"].named_children():
        if isinstance(m, torch.nn.Embedding) and m is not E:
            p = m(torch.arange(ids.shape[1], device=ids.device)[None])
            rows.append((f"E(ids) + {nm}(positions)", *err(hs0, E(ids) + p)))
    return pd.DataFrame(rows, columns=["candidate for hidden_states[0]", "max_abs", "max_rel"])

emb_df = embedding_check(model, parts, ids, hs[0])
with pd.option_context("display.float_format", "{:.2e}".format):
    print(emb_df.to_string(index=False))
E = model.get_input_embeddings()
print("embedding class:", type(E).__name__, "| ‖hidden_states[0]‖ / ‖E.weight[ids]‖ at POS = %.4f"
      % (hs[0][0, pos].norm() / E.weight[ids][0, pos].norm()).item())

token ids  : [464, 3797, 3332, 319, 262, 2603, 780]
tokens     : ['The', 'Ġcat', 'Ġsat', 'Ġon', 'Ġthe', 'Ġmat', 'Ġbecause']
BOS token  : <|endoftext|> | first id is BOS: False
padding    : right (one prompt, no padding)
explicit position POS = 6 'Ġbecause'

len(hidden_states) = 13 for 12 blocks; each entry has shape (1, 7, 768)
candidate for hidden_states[0]  max_abs  max_rel
  raw table rows E.weight[ids] 4.54e+00 3.63e+00
       embedding module E(ids) 4.54e+00 3.63e+00
       E(ids) + wpe(positions) 0.00e+00 0.00e+00
embedding class: Embedding | ‖hidden_states[0]‖ / ‖E.weight[ids]‖ at POS = 1.7113


## 3 · Hooks on the attention and MLP modules, and a numerical check of the guess

The hooks were placed by `collect` on every direct child of every block. The *guess* from section 1a says which two
children are "the writes". Before using it, the cell searches all pairs of children for the pair whose outputs add up to
(block output − block input), and which child receives the mid-block stream (block input + attention write) as its
input. If the guess and the search disagree, the search wins and the cell says so.

In [8]:
def pair_search(rec, L):
    x_in, x_out = rec["block_in"][L], rec["block_out"][L]
    outs = {k: v for k, v in rec["child_out"][L].items() if v.shape == x_in.shape}
    rows = []
    for a, b in itertools.combinations(outs, 2):
        rows.append((a, b, *err(x_out, x_in + outs[a] + outs[b])))
    return pd.DataFrame(rows, columns=["child A", "child B", "max_abs", "max_rel"]).sort_values("max_rel")

def mid_reader(rec, L, attn_name):
    x_mid = rec["block_in"][L] + rec["child_out"][L][attn_name]
    rows = [(k, *err(v, x_mid)) for k, v in rec["child_in"][L].items() if v.shape == x_mid.shape]
    return pd.DataFrame(rows, columns=["child", "max_abs", "max_rel"]).sort_values("max_rel")

def resolve_writers(parts, rec, tol=1e-5):
    ps = pair_search(rec, 0)
    best = ps.iloc[0]
    found = best["max_rel"] < tol
    guess = (parts["attn_name"], parts["mlp_name"])
    if found and set(guess) == {best["child A"], best["child B"]}:
        return guess, ps, "guess confirmed by the pair search"
    if found:
        # keep the order of the block's children: the earlier child is the attention-side write
        order = [nm for nm, _ in parts["children"]]
        w = tuple(sorted([best["child A"], best["child B"]], key=order.index))
        return w, ps, f"guess {guess} REJECTED; the writes are {w}"
    return guess, ps, "no pair of children reproduces the block's write: hooks on children are not enough"

(WA, WM), ps0, verdict = resolve_writers(parts, rec)
with pd.option_context("display.float_format", "{:.2e}".format):
    print("block 0, all pairs of children:\n", ps0.to_string(index=False))
    print("\nVERDICT:", verdict, "-> attention write =", WA, ", MLP write =", WM)
    print("\nwhich child reads the mid-block stream (block input + attention write)?\n", mid_reader(rec, 0, WA).to_string(index=False))

block 0, all pairs of children:
 child A child B  max_abs  max_rel
   attn     mlp 0.00e+00 0.00e+00
   ln_2     mlp 1.48e+01 8.95e-01
   ln_1     mlp 1.50e+01 9.54e-01
   attn    ln_2 9.39e+01 4.05e+00
   ln_1    attn 9.50e+01 4.45e+00
   ln_1    ln_2 9.38e+01 7.70e+00

VERDICT: guess confirmed by the pair search -> attention write = attn , MLP write = mlp

which child reads the mid-block stream (block input + attention write)?
 child  max_abs  max_rel
 ln_2 0.00e+00 0.00e+00
  mlp 1.47e+01 9.50e-01
 ln_1 1.49e+01 9.78e-01
 attn 1.49e+01 9.87e-01


## 4 · Per-block additivity, against the hooks and against `hidden_states`

For every block l: (a) block output − (block input + attention write + MLP write); (b) `hidden_states[l]` versus the
block's input; (c) `hidden_states[l+1]` versus the block's output; (d) `hidden_states[l+1] − hidden_states[l]` versus
attention write + MLP write. A failure is reported, not hidden (card T1).

In [9]:
def additivity(rec, hs, WA, WM):
    rows = []
    for L in range(len(rec["block_in"])):
        a, m = rec["child_out"][L][WA], rec["child_out"][L][WM]
        x_in, x_out = rec["block_in"][L], rec["block_out"][L]
        rows.append(dict(block=L,
                         a_out_vs_in_plus_writes=err(x_out, x_in + a + m)[1],
                         b_hs_l_vs_block_in=err(hs[L], x_in)[1],
                         c_hs_l1_vs_block_out=err(hs[L + 1], x_out)[1],
                         d_hs_diff_vs_writes=err(hs[L + 1] - hs[L], a + m)[1],
                         d_abs=err(hs[L + 1] - hs[L], a + m)[0],
                         stream_norm_max=x_out.norm(dim=-1).max().item()))
    return pd.DataFrame(rows).set_index("block")

add_df = additivity(rec, hs, WA, WM)
with pd.option_context("display.float_format", "{:.2e}".format):
    print("relative errors (max over positions of ‖diff‖/‖reference‖); d_abs = max absolute entry of (d)\n")
    print(add_df)
TOL = 1e-5
bad = {c: add_df.index[add_df[c] > TOL].tolist() for c in ["a_out_vs_in_plus_writes", "b_hs_l_vs_block_in", "c_hs_l1_vs_block_out", "d_hs_diff_vs_writes"]}
print("\nblocks where the relative error exceeds", TOL, ":", bad)
print("max relative error of (a) over all blocks: %.3e" % add_df.a_out_vs_in_plus_writes.max())
print("max relative error of (d) over blocks 0..n-2: %.3e" % add_df.d_hs_diff_vs_writes.iloc[:-1].max())

relative errors (max over positions of ‖diff‖/‖reference‖); d_abs = max absolute entry of (d)

       a_out_vs_in_plus_writes  b_hs_l_vs_block_in  c_hs_l1_vs_block_out  d_hs_diff_vs_writes    d_abs  stream_norm_max
block                                                                                                                  
0                     0.00e+00            0.00e+00              0.00e+00             6.45e-08 1.91e-06         1.33e+02
1                     0.00e+00            0.00e+00              0.00e+00             1.98e-07 7.63e-06         6.36e+02
2                     0.00e+00            0.00e+00              0.00e+00             2.15e-07 4.83e-06         2.56e+03
3                     0.00e+00            0.00e+00              0.00e+00             2.47e-07 4.58e-05         2.75e+03
4                     0.00e+00            0.00e+00              0.00e+00             1.12e-06 1.68e-04         2.90e+03
5                     0.00e+00            0.00e+00              0

## 5 · The last entry of `hidden_states` and the final norm

In [10]:
def final_entry(model, parts, rec, hs, logits):
    last_out = rec["block_out"][-1]
    normed = parts["final_norm"](last_out)
    lm_head = model.get_output_embeddings()
    rows = [("hidden_states[-1] vs output of the last block", *err(hs[-1], last_out)),
            (f"hidden_states[-1] vs {parts['final_norm_name']}(output of the last block)", *err(hs[-1], normed)),
            ("lm_head(hidden_states[-1]) vs model logits", *err(lm_head(hs[-1]), logits))]
    return pd.DataFrame(rows, columns=["comparison", "max_abs", "max_rel"])

fe_df = final_entry(model, parts, rec, hs, out.logits)
with pd.option_context("display.float_format", "{:.2e}".format, "display.max_colwidth", 80):
    print(fe_df.to_string(index=False))
print("\nSo hidden_states[-1] is the final-normed vector that the unembedding reads, not the stream after the last")
print("block; that stream is only available from a hook on the last block (rec['block_out'][-1] here).")

                                         comparison  max_abs  max_rel
      hidden_states[-1] vs output of the last block 3.70e+02 9.86e-01
hidden_states[-1] vs ln_f(output of the last block) 0.00e+00 0.00e+00
         lm_head(hidden_states[-1]) vs model logits 0.00e+00 0.00e+00

So hidden_states[-1] is the final-normed vector that the unembedding reads, not the stream after the last
block; that stream is only available from a hook on the last block (rec['block_out'][-1] here).


## 6 · One explicit position and layer, rebuilt from its writes

`h = hidden_states[LAYER][0, POS]` is rebuilt as `hidden_states[0][0, POS] + Σ_{l < LAYER} (attention write + MLP write)`.
Then the same idea as W9-B on a real stream: the score of `h` along a direction splits over the writers.

In [11]:
LAYER_ = n // 2 if LAYER is None else LAYER
assert 0 <= LAYER_ < n, "LAYER must be < n_layers: the last entry of hidden_states is final-normed (section 5)"
h = hs[LAYER_][0, pos]
writes = [hs[0][0, pos]] + [w for L in range(LAYER_) for w in (rec["child_out"][L][WA][0, pos], rec["child_out"][L][WM][0, pos])]
rebuilt = torch.stack(writes).sum(0)
print(f"position {pos} {str_toks[pos]!r}, after {LAYER_} blocks: d = {h.numel()}, ‖h‖ = {h.norm():.4f}")
print("h vs embedding + all writes before it: max abs %.3e, rel %.3e" % err(h[None], rebuilt[None]))

g = torch.Generator().manual_seed(SEED)
u = torch.randn(h.numel(), generator=g, dtype=DTYPE); u = u / u.norm()        # a random unit direction (no meaning)
shares = torch.stack([u @ w for w in writes])
print("score u·h = %.6f | sum of the %d read-shares = %.6f | |difference| = %.2e"
      % ((u @ h).item(), len(writes), shares.sum().item(), abs((u @ h - shares.sum()).item())))
labels = ["embedding"] + [f"{k}{L}" for L in range(LAYER_) for k in ("attn", "mlp")]
top = sorted(zip(labels, shares.tolist()), key=lambda t: -abs(t[1]))[:6]
print("largest read-shares along u:", [(k, round(v, 4)) for k, v in top])

position 6 'Ġbecause', after 6 blocks: d = 768, ‖h‖ = 77.1585
h vs embedding + all writes before it: max abs 0.000e+00, rel 0.000e+00
score u·h = 7.211561 | sum of the 13 read-shares = 7.211561 | |difference| = 0.00e+00
largest read-shares along u: [('mlp0', 3.314), ('attn0', 1.7538), ('mlp5', 0.9521), ('mlp4', 0.4068), ('mlp2', 0.36), ('mlp1', 0.2769)]


## 7 · Arithmetic on a real stream (card T9)

Random unit direction `u` (seeded; no meaning, and no difference-of-prompts vector, which belongs to a later post).
A read is linear; a steering write `h ↦ h + αu` is a translation (affine): the score moves by exactly α; cosine similarity
divides by norms and is not additive.

In [12]:
s0 = (u @ h).item()
rows = []
for alpha in [0.5, 1.0, 2.0, 5.0, -3.0]:
    s1 = (u @ (h + alpha * u)).item()
    rows.append(dict(alpha=alpha, score_before=s0, score_after=s1, shift=s1 - s0, shift_minus_alpha=s1 - s0 - alpha))
with pd.option_context("display.float_format", "{:.6g}".format):
    print(pd.DataFrame(rows).to_string(index=False))

x1, x2 = hs[LAYER_][0, 0], hs[LAYER_][0, pos]
print("\nread is linear: u·(x1+x2) − (u·x1 + u·x2) = %.2e" % (u @ (x1 + x2) - (u @ x1 + u @ x2)).item())
steer = lambda x, a=2.0: x + a * u
print("steering is affine, not linear: steer(x1+x2) − (steer(x1)+steer(x2)) has norm %.4f (= |α|·‖u‖ = 2)"
      % (steer(x1 + x2) - (steer(x1) + steer(x2))).norm().item())
cos = lambda a, b: (a @ b / (a.norm() * b.norm())).item()
v = 3.0 * u
print("cosine is not additive: cos(h+v, u) = %.4f, cos(h, u) + cos(v, u) = %.4f" % (cos(h + v, u), cos(h, u) + cos(v, u)))

 alpha  score_before  score_after  shift  shift_minus_alpha
   0.5       7.21156      7.71156    0.5                  0
     1       7.21156      8.21156      1                  0
     2       7.21156      9.21156      2                  0
     5       7.21156      12.2116      5                  0
    -3       7.21156      4.21156     -3                  0

read is linear: u·(x1+x2) − (u·x1 + u·x2) = -5.72e-06
steering is affine, not linear: steer(x1+x2) − (steer(x1)+steer(x2)) has norm 2.0000 (= |α|·‖u‖ = 2)
cosine is not additive: cos(h+v, u) = 0.1318, cos(h, u) + cos(v, u) = 1.0935


## 8 · The same checks on tiny randomly initialised models (code path only)

**8a · Qwen3.** Only the `config.json` of `TINY_CONFIG_FROM` is read (no weights); layers and widths are shrunk and the
weights are random (seeded). The numbers say nothing about the real model's stream; they test that the hooks, the
indexing of `hidden_states` and the final-norm handling work for the Qwen3 code in the installed `transformers`.

**8b · Gemma 3 (text).** The installed library's *default* `Gemma3TextConfig()` (nothing downloaded), shrunk the same
way. Its decoder layer applies norms to the sub-layer outputs before the add (see the source printed below), so hooks on
the attention and MLP modules themselves are expected to fail the additivity check; the pair search should find the
modules whose outputs are actually added.

In [13]:
def summary(model, tok_or_none, label, seq_len=8):
    parts_ = find_parts(model)
    if tok_or_none is not None:
        ids_ = tok_or_none(PROMPT, return_tensors="pt").input_ids
    else:
        ids_ = torch.randint(0, model.config.vocab_size, (1, seq_len), generator=torch.Generator().manual_seed(SEED))
    out_, rec_ = collect(model, parts_, ids_)
    hs_ = out_.hidden_states
    n_ = parts_["n_layers"]
    guess = (parts_["attn_name"], parts_["mlp_name"])
    guess_err = max(err(rec_["block_out"][L], rec_["block_in"][L] + rec_["child_out"][L][guess[0]] + rec_["child_out"][L][guess[1]])[1]
                    for L in range(n_))
    (wa, wm), _, verdict_ = resolve_writers(parts_, rec_)
    add_ = additivity(rec_, hs_, wa, wm)
    fe_ = final_entry(model, parts_, rec_, hs_, out_.logits)
    emb_ = embedding_check(model, parts_, ids_, hs_[0])
    E = model.get_input_embeddings()
    return {
        "model": label,
        "blocks / d": f"{n_} / {model.config.hidden_size}",
        "len(hidden_states)": len(hs_),
        "hs[0] vs embed lookup (rel)": f"{emb_.max_rel.min():.1e} ({emb_.iloc[emb_.max_rel.argmin(), 0]})",
        "embedding class": type(E).__name__,
        "hs[0] vs raw table rows E.weight[ids] (rel)": f"{emb_.max_rel[0]:.1e}",
        "‖hs[0]‖/‖E.weight[ids]‖ (first position)": f"{(hs_[0][0, 0].norm() / E.weight[ids_][0, 0].norm()).item():.4f}",
        "guessed writes (attn, mlp)": guess,
        "additivity with guessed writes (max rel)": f"{guess_err:.1e}",
        "pair-search writes": (wa, wm),
        "additivity with found writes (max rel)": f"{add_.a_out_vs_in_plus_writes.max():.1e}",
        "child that reads block input + attention write": mid_reader(rec_, 0, wa).iloc[0, 0],
        "hs[l+1]-hs[l] = writes, l < n-1 (max rel)": f"{add_.d_hs_diff_vs_writes.iloc[:-1].max():.1e}",
        "hs[l+1]-hs[l] = writes, l = n-1 (rel)": f"{add_.d_hs_diff_vs_writes.iloc[-1]:.1e}",
        "hs[-1] vs last block output (rel)": f"{fe_.max_rel[0]:.1e}",
        "hs[-1] vs final_norm(last block output) (rel)": f"{fe_.max_rel[1]:.1e}",
        "lm_head(hs[-1]) vs logits (abs)": f"{fe_.max_abs[2]:.1e}",
        "tie_word_embeddings / shared storage": (getattr(model.config, "tie_word_embeddings", None),
            model.get_output_embeddings().weight.data_ptr() == E.weight.data_ptr()),
        "verdict": verdict_,
    }

results = [summary(model, tok, f"{LOADED} (pretrained)")]

# 8a: tiny random Qwen3 from the config only
try:
    qcfg = AutoConfig.from_pretrained(TINY_CONFIG_FROM, local_files_only=LOCAL_ONLY)
    print("full-size config:", {k: getattr(qcfg, k) for k in ["model_type", "hidden_size", "num_hidden_layers", "num_attention_heads",
          "num_key_value_heads", "head_dim", "intermediate_size", "rms_norm_eps", "tie_word_embeddings", "vocab_size"]})
    d = qcfg.to_dict()
    d.update(num_hidden_layers=4, hidden_size=64, intermediate_size=192, num_attention_heads=4,
             num_key_value_heads=2, head_dim=16, max_window_layers=4)
    d.pop("layer_types", None)
    tiny_cfg = type(qcfg)(**d)
    torch.manual_seed(SEED)
    tiny_q = AutoModelForCausalLM.from_config(tiny_cfg, attn_implementation="eager").to(DTYPE).eval()
    try:
        tiny_tok = AutoTokenizer.from_pretrained(TINY_CONFIG_FROM, local_files_only=LOCAL_ONLY)
    except Exception as e:
        print("tokenizer not available, using random token ids:", type(e).__name__)
        tiny_tok = None
    if tiny_tok is not None:
        t_ids = tiny_tok(PROMPT).input_ids
        print("Qwen3 tokenizer:", tiny_tok.convert_ids_to_tokens(t_ids), "| bos_token:", tiny_tok.bos_token,
              "| padding side:", tiny_tok.padding_side)
    results.append(summary(tiny_q, tiny_tok, "tiny random Qwen3 (config only)"))
except Exception as e:
    print("8a skipped:", type(e).__name__, str(e)[:300])

# 8b: tiny random Gemma 3 text model from the library's default config
try:
    from transformers import Gemma3TextConfig
    gcfg = Gemma3TextConfig(vocab_size=1000, hidden_size=64, intermediate_size=128, num_hidden_layers=4,
                            num_attention_heads=4, num_key_value_heads=2, head_dim=16)
    torch.manual_seed(SEED)
    tiny_g = AutoModelForCausalLM.from_config(gcfg, attn_implementation="eager").to(DTYPE).eval()
    show_source(type(find_parts(tiny_g)["layers"][0]).forward, pattern=r"residual|layernorm\(|self_attn\(|self\.mlp\(")
    show_source(type(tiny_g.get_input_embeddings()).forward)
    results.append(summary(tiny_g, None, "tiny random Gemma 3 text (default config)"))
except Exception as e:
    print("8b skipped:", type(e).__name__, str(e)[:300])

summary_df = pd.DataFrame(results).set_index("model").T
with pd.option_context("display.max_colwidth", 70, "display.width", 220):
    print(summary_df)

full-size config: {'model_type': 'qwen3', 'hidden_size': 2048, 'num_hidden_layers': 28, 'num_attention_heads': 16, 'num_key_value_heads': 8, 'head_dim': 128, 'intermediate_size': 6144, 'rms_norm_eps': 1e-06, 'tie_word_embeddings': True, 'vocab_size': 151936}


Qwen3 tokenizer: ['The', 'Ġcat', 'Ġsat', 'Ġon', 'Ġthe', 'Ġmat', 'Ġbecause'] | bos_token: None | padding side: right
transformers/models/gemma3/modeling_gemma3.py:397:         residual = hidden_states
transformers/models/gemma3/modeling_gemma3.py:399:         hidden_states = self.input_layernorm(hidden_states)
transformers/models/gemma3/modeling_gemma3.py:401:         hidden_states, _ = self.self_attn(
transformers/models/gemma3/modeling_gemma3.py:409:         hidden_states = self.post_attention_layernorm(hidden_states)
transformers/models/gemma3/modeling_gemma3.py:410:         hidden_states = residual + hidden_states
transformers/models/gemma3/modeling_gemma3.py:412:         residual = hidden_states
transformers/models/gemma3/modeling_gemma3.py:413:         hidden_states = self.pre_feedforward_layernorm(hidden_states)
transformers/models/gemma3/modeling_gemma3.py:414:         hidden_states = self.mlp(hidden_states)
transformers/models/gemma3/modeling_gemma3.py:415:         hidden_state

## 9 · Your turn (blank exercise cells)

The W9 numbers are in notebook 01, section 7. Here you can also try them on the real stream `h` above.

**E1 · Add.** Rebuild `hidden_states[3]` at one position from `hidden_states[0]` and the hooked writes of blocks 0–2.

In [14]:
# E1: your code here

**E2 · Who hears whom.** For a random unit `u`, which writer contributes most to `u·h`? Is any writer nearly inaudible?

In [15]:
# E2: your code here

**E3 · Project.** Score every position of `hidden_states[LAYER_]` on three random unit directions. Do the scores separate
any tokens?

In [16]:
# E3: your code here

**E4 · Steer.** Add `α·u` to `h` and check the score moves by α. Add a vector orthogonal to `u`: by how much does the score move?

In [17]:
# E4: your code here

**E5 · Rotate.** Draw a random orthogonal matrix Q (`torch.linalg.qr`), rotate `h`, every write and `u`. Are the
read-shares unchanged? What does that say about "coordinate 17 of the stream"?

In [18]:
# E5: your code here